Phase 2: Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_csv('../data/processed/nsei_cleaned.csv', parse_dates=['Date'], index_col='Date')
print(f"Loaded cleaned data: {df.shape}")
display(df.head())

2.1 Lag Features

In [ ]:
# Lagged returns (past 1, 2, 3, 5, 10 days)
for lag in [1, 2, 3, 5, 10]:
    df[f'Return_lag{lag}'] = df['Return'].shift(lag)

# Lagged absolute returns (proxy for recent volatility)
for lag in [1, 2, 3, 5]:
    df[f'AbsReturn_lag{lag}'] = df['Return'].abs().shift(lag)

print("Lag features created.")
display(df[['Return', 'Return_lag1', 'Return_lag2', 'AbsReturn_lag1']].head(10))

2.2 Rolling Volatility Features

In [ ]:
for window in [5, 10, 20]:
    df[f'Vol_rolling_{window}'] = df['Return'].rolling(window=window).std()

# Also add rolling mean of absolute returns (alternative vol proxy)
df['AbsReturn_roll_5'] = df['Return'].abs().rolling(window=5).mean()

print("Rolling volatility features created.")
display(df[['Vol_rolling_5', 'Vol_rolling_10', 'Vol_rolling_20', 'AbsReturn_roll_5']].tail())

2.3 Technical Indicators

In [ ]:
# 2.3a: Moving Averages
for window in [5, 20, 50]:
    df[f'MA_{window}'] = df['Close'].rolling(window=window).mean()
    
# Distance of price from MA (normalized) — often more informative than raw MA
df['Close_MA20_ratio'] = df['Close'] / df['MA_20']
df['Close_MA50_ratio'] = df['Close'] / df['MA_50']

print("Moving averages created.")
display(df[['Close', 'MA_5', 'MA_20', 'MA_50', 'Close_MA20_ratio']].tail())

In [ ]:
# 2.3b: RSI 14
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0.0)
loss = -delta.where(delta < 0, 0.0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss.replace(0, np.nan)
df['RSI_14'] = 100 - (100 / (1 + rs))

print("RSI(14) created.")
display(df[['Close', 'RSI_14']].tail())

In [ ]:
# 2.3c: MACD
ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
ema_26 = df['Close'].ewm(span=26, adjust=False).mean()

df['MACD'] = ema_12 - ema_26
df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_hist'] = df['MACD'] - df['MACD_signal']

print("MACD created.")
display(df[['Close', 'MACD', 'MACD_signal', 'MACD_hist']].tail())

In [ ]:
# 2.3d: ATR 14
high_low = df['High'] - df['Low']
high_close = (df['High'] - df['Close'].shift(1)).abs()
low_close = (df['Low'] - df['Close'].shift(1)).abs()

true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df['ATR_14'] = true_range.rolling(window=14).mean()

print("ATR(14) created.")
display(df[['High', 'Low', 'Close', 'ATR_14']].tail())

2.4 Volume-Based Features

In [ ]:
df['Volume_MA_5'] = df['Volume'].rolling(window=5).mean()
df['Volume_MA_20'] = df['Volume'].rolling(window=20).mean()
df['Volume_ratio'] = df['Volume'] / df['Volume_MA_20']

print("Volume-based features created.")
display(df[['Volume', 'Volume_MA_5', 'Volume_MA_20', 'Volume_ratio']].tail())

2.5 Define Target Variable
Target = next-day 5-day forward volatility.

We compute the std of returns over the next 5 days (t+1 to t+5). We use shift(-5) on the rolling std, so at time *t* the value is the std of returns at [t+1, t+5]. This is what the model will try to predict at time *t*.

In [ ]:
# Rolling std of returns, then shift backwards so it aligns with the *current* row
forward_window = 5
df['Target_Volatility'] = df['Return'].rolling(window=forward_window).std().shift(-forward_window)

print(f"Target = {forward_window}-day forward volatility")
display(df[['Return', 'Target_Volatility']].tail(10))